# 04. 멀티에이전트 Supervisor 패턴

## 학습 목표
- Supervisor 패턴의 개념을 이해한다 (중앙 관리자가 워커 에이전트에게 작업 위임)
- `with_structured_output`과 Pydantic으로 라우팅 로직을 구현할 수 있다
- 여러 에이전트를 StateGraph로 연결하는 멀티에이전트 워크플로우를 구성할 수 있다
- 다양한 질문으로 에이전트 간 협업 흐름을 확인할 수 있다

## 1. 환경 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env 파일에 설정되어 있는지 확인하세요!"

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.tools import tool

llm = ChatOpenAI(model="gpt-5-mini")

print("환경 설정 완료!")

## 2. 워커 에이전트 3개 정의

각 에이전트는 특화된 역할을 수행합니다.

- **Search Agent**: 정보 검색 담당
- **Analysis Agent**: 데이터 분석 담당
- **Writer Agent**: 보고서 작성 담당

In [ ]:
from langgraph.prebuilt import create_react_agent

# === Search Agent Tools ===
@tool
def search_web(query: str) -> str:
    """웹에서 최신 정보를 검색합니다."""
    mock_data = {
        "AI 시장": "2025년 글로벌 AI 시장 규모는 약 3,000억 달러로 추정됩니다. 생성형 AI가 가장 빠르게 성장하는 분야입니다.",
        "LLM": "GPT-5, Claude 4, Gemini 2.0 등 최신 LLM이 멀티모달, 에이전트 기능을 강화하고 있습니다.",
        "에이전트": "AI 에이전트 시장이 급성장 중이며, 기업용 에이전트 솔루션이 주목받고 있습니다.",
    }
    for key, value in mock_data.items():
        if key in query:
            return value
    return f"'{query}' 검색 결과: AI 기술이 빠르게 발전하며 다양한 산업에 적용되고 있습니다."

@tool
def search_news(topic: str) -> str:
    """최신 뉴스를 검색합니다."""
    return f"[뉴스] '{topic}' 관련 최신 뉴스: AI 에이전트 기술이 2025년 주요 트렌드로 부상하고 있습니다."

# === Analysis Agent Tools ===
@tool
def analyze_data(data: str) -> str:
    """데이터를 분석하고 인사이트를 도출합니다."""
    return f"분석 결과: {data}에 대한 분석을 완료했습니다. 핵심 트렌드는 '자동화', '멀티모달', '에이전트'입니다."

@tool
def compare_options(option_a: str, option_b: str) -> str:
    """두 옵션을 비교 분석합니다."""
    return f"비교 분석: '{option_a}' vs '{option_b}' - 각각의 장단점을 분석했습니다. {option_a}는 범용성이 높고, {option_b}는 특화된 성능이 강점입니다."

# === Writer Agent Tools ===
@tool
def write_report(topic: str, content: str) -> str:
    """주제와 내용을 기반으로 보고서를 작성합니다."""
    return f"""=== 보고서: {topic} ===
\n1. 개요: {topic}에 대한 종합 분석 보고서
2. 주요 내용: {content[:200]}
3. 결론: 해당 분야는 지속적인 성장이 예상됩니다.
========================"""

@tool
def format_summary(text: str) -> str:
    """텍스트를 깔끔한 요약 형식으로 정리합니다."""
    return f"[요약]\n{text[:300]}\n[끝]"

# 워커 에이전트 생성
search_agent = create_react_agent(
    llm,
    [search_web, search_news],
    prompt="당신은 정보 검색 전문가입니다. 주어진 주제에 대해 관련 정보를 검색하여 제공하세요. 한국어로 답변하세요."
)

analysis_agent = create_react_agent(
    llm,
    [analyze_data, compare_options],
    prompt="당신은 데이터 분석 전문가입니다. 주어진 데이터를 분석하고 인사이트를 도출하세요. 한국어로 답변하세요."
)

writer_agent = create_react_agent(
    llm,
    [write_report, format_summary],
    prompt="당신은 보고서 작성 전문가입니다. 주어진 정보를 바탕으로 깔끔한 보고서를 작성하세요. 한국어로 답변하세요."
)

print("워커 에이전트 3개 생성 완료!")
print("  - search_agent: 정보 검색")
print("  - analysis_agent: 데이터 분석")
print("  - writer_agent: 보고서 작성")

## 3. Supervisor 라우팅 로직

Supervisor는 사용자의 요청을 분석하고, 적절한 워커 에이전트에게 작업을 위임합니다.

`with_structured_output`과 Pydantic을 사용하여 구조화된 라우팅 결정을 합니다.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# Supervisor의 라우팅 결정 스키마
class RouteDecision(BaseModel):
    """Supervisor가 내리는 라우팅 결정"""
    next_agent: Literal["search", "analysis", "writer", "FINISH"] = Field(
        description="다음에 작업할 에이전트. 모든 작업이 완료되면 FINISH"
    )
    instruction: str = Field(
        description="선택된 에이전트에게 전달할 구체적인 지시사항"
    )

# Supervisor LLM (structured output)
supervisor_llm = llm.with_structured_output(RouteDecision)

# 테스트: Supervisor 라우팅 결정
supervisor_prompt = """당신은 팀 리더(Supervisor)입니다. 사용자의 요청과 현재까지의 작업 결과를 보고,
다음에 어떤 에이전트가 작업해야 하는지 결정하세요.

팀원:
- search: 웹 검색으로 정보를 수집합니다
- analysis: 수집된 정보를 분석합니다
- writer: 분석 결과를 보고서로 작성합니다
- FINISH: 모든 작업이 완료되었을 때

일반적인 흐름: search -> analysis -> writer -> FINISH
단순 질문은 바로 적절한 에이전트에게 보내세요."""

test_messages = [
    SystemMessage(content=supervisor_prompt),
    HumanMessage(content="AI 에이전트 시장 동향에 대한 보고서를 작성해주세요.")
]

decision = supervisor_llm.invoke(test_messages)
print(f"다음 에이전트: {decision.next_agent}")
print(f"지시사항: {decision.instruction}")

## 4. StateGraph로 멀티에이전트 워크플로우 구성

Supervisor + 워커 에이전트를 하나의 StateGraph로 연결합니다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END, add_messages

# 멀티에이전트 State 정의
class MultiAgentState(TypedDict):
    messages: Annotated[list, add_messages]  # 전체 대화 히스토리
    next_agent: str  # 다음 에이전트
    agent_outputs: dict  # 각 에이전트의 출력 저장

# Supervisor 노드
def supervisor_node(state: MultiAgentState) -> dict:
    """Supervisor가 다음 에이전트를 결정합니다."""
    # 에이전트 출력 요약
    agent_outputs = state.get("agent_outputs", {})
    context = ""
    if agent_outputs:
        context = "\n\n현재까지의 작업 결과:\n"
        for agent_name, output in agent_outputs.items():
            context += f"- {agent_name}: {output[:200]}\n"
    
    messages = [
        SystemMessage(content=supervisor_prompt + context),
        *state["messages"]
    ]
    
    decision = supervisor_llm.invoke(messages)
    print(f"  [Supervisor] -> {decision.next_agent}: {decision.instruction}")
    
    # 지시사항을 메시지에 추가
    return {
        "next_agent": decision.next_agent,
        "messages": [AIMessage(content=f"[Supervisor -> {decision.next_agent}] {decision.instruction}")]
    }

# 워커 노드 (각 에이전트 실행)
def search_node(state: MultiAgentState) -> dict:
    """Search Agent 실행"""
    result = search_agent.invoke({"messages": state["messages"]})
    output = result["messages"][-1].content
    print(f"  [Search Agent] 완료")
    
    agent_outputs = state.get("agent_outputs", {})
    agent_outputs["search"] = output
    
    return {
        "messages": [AIMessage(content=f"[Search 결과] {output}")],
        "agent_outputs": agent_outputs
    }

def analysis_node(state: MultiAgentState) -> dict:
    """Analysis Agent 실행"""
    result = analysis_agent.invoke({"messages": state["messages"]})
    output = result["messages"][-1].content
    print(f"  [Analysis Agent] 완료")
    
    agent_outputs = state.get("agent_outputs", {})
    agent_outputs["analysis"] = output
    
    return {
        "messages": [AIMessage(content=f"[Analysis 결과] {output}")],
        "agent_outputs": agent_outputs
    }

def writer_node(state: MultiAgentState) -> dict:
    """Writer Agent 실행"""
    result = writer_agent.invoke({"messages": state["messages"]})
    output = result["messages"][-1].content
    print(f"  [Writer Agent] 완료")
    
    agent_outputs = state.get("agent_outputs", {})
    agent_outputs["writer"] = output
    
    return {
        "messages": [AIMessage(content=f"[최종 보고서]\n{output}")],
        "agent_outputs": agent_outputs
    }

# 라우팅 함수
def route_to_agent(state: MultiAgentState) -> str:
    """Supervisor 결정에 따라 다음 노드로 라우팅"""
    next_agent = state["next_agent"]
    if next_agent == "FINISH":
        return END
    return next_agent

# 그래프 구성
builder = StateGraph(MultiAgentState)

# 노드 추가
builder.add_node("supervisor", supervisor_node)
builder.add_node("search", search_node)
builder.add_node("analysis", analysis_node)
builder.add_node("writer", writer_node)

# 엣지 추가
builder.add_edge(START, "supervisor")

# Supervisor -> 각 에이전트 또는 종료 (조건부 엣지)
builder.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {"search": "search", "analysis": "analysis", "writer": "writer", END: END}
)

# 각 에이전트 -> Supervisor (결과 보고)
builder.add_edge("search", "supervisor")
builder.add_edge("analysis", "supervisor")
builder.add_edge("writer", "supervisor")

# 컴파일 (무한 루프 방지를 위한 recursion_limit 설정)
multi_agent_graph = builder.compile()

print("멀티에이전트 그래프 구성 완료!")

## 5. 그래프 시각화

In [ ]:
# ASCII 시각화
print("=== 멀티에이전트 그래프 구조 ===")
multi_agent_graph.get_graph().print_ascii()
print()

# Mermaid 시각화
try:
    from IPython.display import Image, display
    img = multi_agent_graph.get_graph().draw_mermaid_png()
    display(Image(img))
except Exception as e:
    print(f"Mermaid PNG 생성 실패: {e}")
    print("\nMermaid 텍스트:")
    print(multi_agent_graph.get_graph().draw_mermaid())

## 6. 테스트: 다양한 질문으로 에이전트 흐름 확인

In [ ]:
# 테스트 1: 복합 요청 (search -> analysis -> writer -> FINISH)
print("=" * 60)
print("테스트 1: AI 시장 동향 보고서 작성")
print("=" * 60)

result = multi_agent_graph.invoke(
    {
        "messages": [HumanMessage(content="AI 에이전트 시장 동향을 조사하고 분석한 후, 보고서로 작성해주세요.")],
        "next_agent": "",
        "agent_outputs": {}
    },
    config={"recursion_limit": 20}
)

# 최종 결과 출력
print("\n=== 최종 결과 ===")
print(result["messages"][-1].content)

In [ ]:
# 테스트 2: 단순 검색 요청
print("=" * 60)
print("테스트 2: 단순 검색")
print("=" * 60)

result = multi_agent_graph.invoke(
    {
        "messages": [HumanMessage(content="최신 LLM 동향을 검색해주세요.")],
        "next_agent": "",
        "agent_outputs": {}
    },
    config={"recursion_limit": 20}
)

print("\n=== 최종 결과 ===")
print(result["messages"][-1].content)

## 7. 실습 과제: 에이전트 추가/수정

### 요구사항
1. 새로운 워커 에이전트를 1개 이상 추가하세요 (예: 번역 에이전트, QA 에이전트 등)
2. Supervisor의 라우팅 로직을 수정하여 새 에이전트를 포함하세요
3. StateGraph를 재구성하세요
4. 새 에이전트가 작동하는 시나리오로 테스트하세요

In [ ]:
# 여기에 실습 코드를 작성하세요

# 1) 새 워커 에이전트 정의


# 2) RouteDecision 수정 (새 에이전트 추가)


# 3) Supervisor 프롬프트 수정


# 4) StateGraph 재구성


# 5) 테스트
